# Notebook #02 — Data Engineering (Silver Layer — 24 `CLEANED_FIELDS` & Data Contracts)

### Các bước thực hiện chính
1. **Load Bronze Snapshots** (`93,081` bản ghi chuẩn 12 cột).
2. **Data Cleaning & IATA Mapping:** Chuẩn hóa mã sân bay IATA (`SGN, HAN, DAD, CXR, PQC...`), chuẩn hóa số hiệu chuyến bay (`VN0210 -> VN210`).
3. **Modulo 1440 Rollover:** Xử lý chuyến bay cất cánh/hạ cánh qua mốc 00:00 (ví dụ `23:50 -> 00:15 = +25.0m`).
4. **3-Tier SHA-256 Deduplication:** Khử bản sao y hệt (`raw_hash`) $	o$ trùng sự kiện (`event_hash`) $	o$ gộp theo `leg_key`, chọn snapshot chốt sau cùng.
5. **Weather Integration & Airport Load:** Ghép lượng mưa, gió giật, tầm nhìn, WMO code từ Open-Meteo và đếm mật độ bay theo giờ.
6. **Excluded Outcome Segregation & Silver Contracts:** Tách riêng ca `diverted`/`unknown` sang `excluded_outcomes.csv`, bảo toàn `delay_min = NaN` cho chuyến hủy, và kiểm chứng Pydantic `validate_silver_contract`.

In [1]:
import json
from pathlib import Path
import pandas as pd

from flight_delay.common.constants import CLEANED_FIELDS, FILE_EXCLUDED, FILE_SILVER
from flight_delay.data_engineering.contracts import validate_silver_contract
from flight_delay.data_engineering.transformation import calculate_delay

# Verify Modulo 1440 rollover edge cases
assert calculate_delay("23:50", "", "00:15", "Departed") == 25.0
assert calculate_delay("00:10", "", "23:55", "Departed") == -15.0
print("Modulo 1440 rollover checks passed (+25.0m across midnight, -15.0m early across midnight).")

silver_df = pd.read_csv(FILE_SILVER, low_memory=False)
excl_df = pd.read_csv(FILE_EXCLUDED, low_memory=False)
ok_silver, errs = validate_silver_contract(silver_df)
assert ok_silver, f"Silver contract error: {errs}"
assert silver_df["leg_key"].is_unique, "leg_key must be unique!"
assert silver_df.loc[silver_df["status_clean"] == "cancelled", "delay_min"].isna().all()

qual = json.loads(Path("reports/data_quality/silver_quality_report.json").read_text(encoding="utf-8"))
print(f"Silver valid legs: {len(silver_df):,} | Columns present: {len(silver_df.columns)} (24 CLEANED_FIELDS covered)")
print(f"Excluded ambiguous rows: {len(excl_df)} -> {excl_df['status_clean'].value_counts().to_dict()}")
print("3-Class Outcome Distribution:", silver_df["status_clean"].value_counts().to_dict())
print("Deduplication Stats:", json.dumps(qual["deduplication_stats"], indent=2))

Modulo 1440 rollover checks passed (+25.0m across midnight, -15.0m early across midnight).
Silver valid legs: 13,570 | Columns present: 24 (24 CLEANED_FIELDS covered)
Excluded ambiguous rows: 40 -> {'unknown': 29, 'diverted': 11}
3-Class Outcome Distribution: {'on_time': 9904, 'delayed': 3571, 'cancelled': 95}
Deduplication Stats: {
  "raw_snapshots": 68250,
  "exact_duplicates": 10,
  "after_exact_dedup": 68240,
  "after_side_dedup": 19387,
  "unit": "leg",
  "after_leg_dedup": 13610,
  "legs_seen_both_sides": 5264,
  "legs_with_t1h_snapshot": 7658,
  "legs_with_t3h_snapshot": 7177,
  "legs_with_t6h_snapshot": 5142
}
